In [5]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("./all-MiniLM-L6-v2")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [7]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [8]:
results[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'Module 7: Streaming',
 'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
 'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
 'doc_id': '5ca6890c1a'}

In [11]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector, 
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5)

results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
if os.getenv("ANTHROPIC_API_KEY") is None:
    raise ValueError("ANTHROPIC_API_KEY environment variable not set")
else:
    print("ANTHROPIC_API_KEY is set correctly")

anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


ANTHROPIC_API_KEY is set correctly


In [ ]:
from rag_helper_anthropic import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=anthropic_client,
)


In [15]:
vector_assistant.rag("the program has already begun, can I still sign up?")

('Based on the context provided, **yes, you can still join!**\n\nHowever, there is an important condition to keep in mind: **if you want to receive a certificate, you need to submit your project while submissions are still being accepted.**\n\nAlso worth noting, you can start learning and submitting homework without any formal registration, as long as the submission forms are still open.',
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=534, output_tokens=81, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [16]:
vs_index.close()